In [ ]:
import pandas as pd

# CSV 파일 읽기
df = pd.read_csv("../../data/DieCasting_Quality_Raw_Data.csv", header=[0,1])

# 상위 5행 확인
df.head()

In [ ]:
df.describe

In [ ]:
# 1.변수 'Biscuit_Thickness ', 'Clamping_Force ', ' Pressure_Rise_Time' 앞 뒤 공백 제거
df.columns = pd.MultiIndex.from_tuples(
    [(col[0].strip(), col[1].strip()) for col in df.columns]
)

print(df.columns.tolist())

In [ ]:
# 2.factory_cols 결측치 중앙값 처리
factory_cols = [
    'Factory_Temp',
    'Factory_Temp_Min',
    'Factory_Temp_Max',
    'Factory_Humidity',
    'Factory_Humidity_Min',
    'Factory_Humidity_Max'
]

for col in factory_cols:
    df[('Sensor', col)] = df[('Sensor', col)].fillna(
        df[('Sensor', col)].median()
    )

df.isnull().sum().sum()
print("중앙값으로 처리 완료!")

In [ ]:
for col in df.columns:
    if "Defect" in col[1]:
        print(col)

In [ ]:
# defect 관련 컬럼 후보 찾기 (대소문자 무시 + 양쪽 공백 제거 + 두 레벨 모두 탐색)
hits = []
for a, b in df.columns:
    a2 = str(a).strip().lower()
    b2 = str(b).strip().lower()
    if ("defect" in a2) or ("defect" in b2):
        hits.append((a, b))

print("defect 포함 컬럼:", hits)

In [ ]:
# 3. Defects 컬럼 1을 초과하는 값 1(불량)으로 대체
defect_cols = [col for col in df.columns if col[0] == "Defects"]

print("불량이 포함된 Defects 컬럼 수:", len(defect_cols))

# 값 1 초과 → 1
df[defect_cols] = df[defect_cols].clip(upper=1)

print("불량값 1대체")

In [ ]:
df[('Process','Product_Type')] = 1

In [ ]:
# 결측치 처리 확인
df.isna().sum()

[이상치 분석]

In [ ]:
df.describe()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TARGET_GROUPS = ["Process"]

# 보통 제외하는 컬럼 (식별자/순번/범주형)
EXCLUDE_SUBCOLS = {"id", "Shot", "Product_Type"}

target_cols = [
    c for c in df.columns
    if c[0] in TARGET_GROUPS and c[1] not in EXCLUDE_SUBCOLS
]


# 숫자형으로 강제 변환(문자 섞인 컬럼 대비)
df_num = df[target_cols].apply(pd.to_numeric, errors="coerce")

print("분석 대상 컬럼 수:", df_num.shape[1])
print("전체 결측 비율(%) 평균:", (df_num.isna().mean().mean() * 100).round(2))


# 1) IQR 기반 이상치 탐지
Q1 = df_num.quantile(0.25)
Q3 = df_num.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

is_out_iqr = (df_num.lt(lower) | df_num.gt(upper))  # DataFrame[bool]

iqr_outlier_count = is_out_iqr.sum().sort_values(ascending=False)
iqr_outlier_rate = (is_out_iqr.mean() * 100).sort_values(ascending=False)

iqr_summary = pd.DataFrame({
    "이상치 갯수": iqr_outlier_count,
    "이상치 비율(%)": iqr_outlier_rate.round(3),
    "하한선": lower,
    "상한선": upper,
    "Q1(25%)": Q1,
    "Q3(75%)": Q3,
    "IQR": IQR
}).sort_values("이상치 비율(%)", ascending=False)

print("\n[IQR 이상치 TOP 15]")
display(iqr_summary.head(15))


In [ ]:
top9_cols = iqr_summary.head(9).index

for col in top9_cols:
    
    lower_val = lower[col]
    upper_val = upper[col]
    
    outliers = df[(df[col] < lower_val) | (df[col] > upper_val)][col]
    
    mean_val = df[col].mean()
    
    print("="*40)
    print(col)
    
    print("이상치 개수:", len(outliers))
    print("최소:", outliers.min())
    print("최대:", outliers.max())
    
    print("평균 대비 최소:", (outliers.min()-mean_val)/mean_val*100, "%")
    print("평균 대비 최대:", (outliers.max()-mean_val)/mean_val*100, "%")

In [ ]:
top9_cols = iqr_summary.head(9).index

for col in top9_cols:
    lower_val = lower[col]
    upper_val = upper[col]

    s = df[col]
    mean_val = s.mean()

    # 전체 min/max
    overall_min = s.min()
    overall_max = s.max()

    # 이상치 마스크
    low_mask  = s < lower_val
    high_mask = s > upper_val
    out_mask  = low_mask | high_mask

    outliers = s[out_mask]

    print("="*60)
    print(col)

    # ✅ min — mean — max 형태로 출력
    print(f"전체 범위: min={overall_min:.6g}  | mean={mean_val:.6g} |  max={overall_max:.6g}")


Velocity_1	   :     1차 사출 속도	


Velocity_2	   :     2차 사출 속도	


Velocity_3	   :     3차 사출 속도	


High_Velocity	:    고속 사출 속도	


Cylinder_Pressure :	실린더 압력	


Pressure_Rise_Time :	압력 상승 시간	


Rapid_Rise_Time	 :   급격 상승 시간	


Biscuit_Thickness :	비스킷 두께	


Cycle_Time	   :     사이클 타임

In [ ]:
# 2) Z-score 기반 이상치 탐지(보조 지표)
# 분포가 심하게 치우친 변수는 Z-score가 과민/둔감할 수 있어 "참고용"
z = (df_num - df_num.mean()) / df_num.std(ddof=0)
Z_TH = 3.0
is_out_z = z.abs() > Z_TH

z_outlier_count = is_out_z.sum().sort_values(ascending=False)
z_outlier_rate = (is_out_z.mean() * 100).sort_values(ascending=False)

z_summary = pd.DataFrame({
    "outlier_count": z_outlier_count,
    "outlier_rate_%": z_outlier_rate.round(3),
}).sort_values("outlier_rate_%", ascending=False)

print("\n[Z-score(|z|>3) 이상치 TOP 15]")
display(z_summary.head(15))

In [ ]:
# -----------------------------
# 3) 변수 1개 골라서 '진짜 이상치' 눈으로 확인(추천 함수)
# -----------------------------
def outlier_report_one(col, method="iqr"):
    """
    col: 멀티인덱스 컬럼 튜플 예) ('Process','Velocity_1')
    method: 'iqr' or 'z'
    """
    s = pd.to_numeric(df[col], errors="coerce")

    if method == "iqr":
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo = q1 - 1.5*iqr
        hi = q3 + 1.5*iqr
        mask = (s < lo) | (s > hi)
        title = f"{col} | IQR outliers: {mask.sum()} ({(mask.mean()*100):.2f}%)  bounds=({lo:.3g},{hi:.3g})"
    else:
        zz = (s - s.mean()) / s.std(ddof=0)
        mask = zz.abs() > Z_TH
        title = f"{col} | Z>|{Z_TH}| outliers: {mask.sum()} ({(mask.mean()*100):.2f}%)"

    # 상/하위 극단값 확인
    extremes = pd.concat([s.nsmallest(5), s.nlargest(5)]).dropna()

    print(title)
    print("\n[극단값 10개(작은 5 + 큰 5)]")
    print(extremes)

    # 시각화: 히스토그램 + 박스플롯(각각)
    plt.figure(figsize=(7,4))
    plt.hist(s.dropna(), bins=40)
    plt.title(f"Histogram: {col}")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(4,4))
    plt.boxplot(s.dropna(), vert=True)
    plt.title(f"Boxplot: {col}")
    plt.tight_layout()
    plt.show()

# -----------------------------
# 4) 이상치가 많은 변수 하나 바로 찍어보기(예시)
# -----------------------------
# IQR 기준 TOP 1 변수 자동 선택
top1_col = iqr_summary.index[0]
outlier_report_one(top1_col, method="iqr")

[불량률 분석]

In [ ]:
# product_type 별 데이터 분리
product_cols = [c for c in df.columns if str(c[1]).strip().lower() == "product_type"]
if not product_cols:
    raise ValueError("Product_Type 컬럼을 찾지 못했습니다. df.columns를 확인하세요.")

product_col = product_cols[0]
print("Product_Type 컬럼:", product_col)
print(df[product_col].value_counts(dropna=False))

In [ ]:
# Product_Type별 불량 컬럼 불량률 확인
product_col = ('Process', 'Product_Type')

defect_rate = df.groupby(product_col)[defect_cols].mean()

print(defect_rate.head())

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(18,6))

defect_rate.T.plot(kind='bar')

plt.title("Product Type별 불량률")
plt.ylabel("Defect Rate")
plt.xlabel("Defect Type")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

top_defects = defect_rate.mean().sort_values(ascending=False).head(10).index

plt.figure(figsize=(18,6))
defect_rate[top_defects].T.plot(kind='bar')

plt.title("Product Type별 불량률 Top 10")
plt.ylabel("Defect Rate")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 센서 변수
sensor_cols = [
    "Melting_Furnace_Temp", "Air_Pressure", "Coolant_Temp",
    "Coolant_Pressure", "Factory_Temp", "Factory_Humidity"
]

plt.figure(figsize=(10,5))
df[sensor_cols].boxplot(rot=90)
plt.title("Sensor Variables Boxplot")
plt.show()